In [ ]:
import logging
from importlib import reload
import os

import torch
import torch.nn as nn
import cv2
import numpy as np
from tqdm import trange

from pathlib import Path

from marmopose.version import __version__ as marmopose_version
from marmopose.config import Config
from marmopose.processing.prediction import Predictor
import matplotlib.pyplot as plt
import matplotlib.patches as patches


logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(name)s - %(message)s')
logger = logging.getLogger(__name__)

logger.info(f'MarmoPose version: {marmopose_version}')

In [ ]:
from marmopose.calibration.cameras import CameraGroup
from marmopose.utils.data_io import load_points_bboxes_2d_h5
dir = 'TestHomeWithEtho1.1'
points_2d, bboxes = load_points_bboxes_2d_h5(f"/scratch/VideoTracking/Videos/{dir}/Output/points_2d/original.h5", [f"output{i+1}" for i in range(6)])


In [ ]:
input_dir = f'/srv/MarmOT/VideoTracking/Videos/{dir}/Input'
vidcaps = [cv2.VideoCapture(os.path.join(input_dir,f'output{i}.mp4')) for i in range(1,7)]

In [ ]:
config_path = '/scratch/VideoTracking/MarmoPose/configs/default.yaml'

config = Config(
    config_path=config_path,
    
    n_tracks=1,
    project='../demos/single',
)
print(config.animal['bodyparts'])
print(points_2d.shape)
print(points_2d[:,0,750,:3])

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.clf()
f = 5010
for i, vidcap in enumerate(vidcaps):
    vidcap.set(cv2.CAP_PROP_POS_FRAMES, f)
    success, frame = vidcap.read()
    print(success)
    if success:
        plt.imshow(frame[:,:,::-1])
        plt.scatter(points_2d[i,0,f,:,0],points_2d[i,0,f,:,1],s= 5)
        plt.axis('off')
        # plt.xlim((0,1920))
        # plt.ylim((1080,0))
        plt.show()


In [ ]:
print(camera_group)

In [ ]:
from tqdm import trange
from marmopose.calibration.cameras import CameraGroup
cams_idx = np.array([4,5])

camera_group = CameraGroup.load_from_json(f"/scratch/VideoTracking/Videos/{dir}/Calib/camera_params.json").subset_cameras_names([f'output{i}' for i in cams_idx + 1])
def triangulate_frame(points_with_score_2d: np.ndarray, ransac=True):
    """
    Args:
        points_with_score_2d: (n_cams, n_bodyparts, (x, y, score))
    
    Returns:
        points_3d: (n_bodyparts, (x, y, z))
    """

    if ransac:
        points_3d = camera_group.triangulate_ransac(points_with_score_2d, undistort=True)
    else:
        points_3d = camera_group.triangulate(points_with_score_2d, undistort=True)
        
    return points_3d

n_cams, n_frames, n_bodyparts, n_dim = points_2d[:,0,...].shape

gt_points_3d = np.full((n_frames, n_bodyparts, 3), np.nan)

for frame_idx in trange(n_frames, ncols=100, desc='Triangulating... ', unit='frames'):
    gt_point_3d = triangulate_frame(points_2d[cams_idx,0,frame_idx,...], ransac=True) 
    gt_points_3d[frame_idx] = gt_point_3d


In [ ]:
for c in camera_group.cameras:
    print(c.name)

In [ ]:
%matplotlib widget


f = 5030
from marmopose.calibration.cameras import CameraGroup
from marmopose.utils.data_io import load_points_3d_h5
from mpl_toolkits.mplot3d import Axes3D
points_3d = load_points_3d_h5(f"/scratch/VideoTracking/Videos/{dir}/Output/points_3d/original.h5")
xlim, ylim, zlim, _ = config.visualization['room_dimensions']
# xlim, ylim, zlim, _ = (100,100,100,100)
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
for bodyparts in config.visualization['skeleton'][::-1]:
    idx_bodyparts = []
    for bodypart in bodyparts:
        idx_bodyparts.append(config.animal['bodyparts'].index(bodypart))
    ax.plot(gt_points_3d[f,idx_bodyparts,0],gt_points_3d[f,idx_bodyparts,1],gt_points_3d[f,idx_bodyparts,2], marker = 'o', ms=3)
# ax.set_xlim((0,xlim))
# ax.set_ylim((0,ylim))
# ax.set_zlim((0,zlim))
ax.set_title('Triangulation')
fig.show()

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
for bodyparts in config.visualization['skeleton'][::-1]:
    idx_bodyparts = []
    for bodypart in bodyparts:
        idx_bodyparts.append(config.animal['bodyparts'].index(bodypart))
    ax.plot(points_3d[0,f,idx_bodyparts,0],points_3d[0,f,idx_bodyparts,1],points_3d[0,f,idx_bodyparts,2], marker = 'o', ms=3)
ax.set_xlim((0,xlim))
ax.set_ylim((0,ylim))
ax.set_zlim((0,zlim))
ax.set_title('Model')
fig.show()


In [ ]:
print(points_3d[0,750,:,:])